# 01 - Preprocessing Pipeline (v2)
## Interview Performance Analyzer

**Dataset:** 2,011 rows x 88 columns (updated with 14 new audio features)
**Objective:** Data inspection, quality checks, type handling, missing value handling,
invalid value handling, duplicate checks, and save clean processed data.

**NOT included:** Feature engineering, PCA, model training, NLP preprocessing, scaling.

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Configuration

In [2]:
# Configuration
_cwd = Path.cwd()
PROJECT_ROOT = None
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "data" / "features" / "merged_features.csv").exists():
        PROJECT_ROOT = _p
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/features/merged_features.csv")

DATA_RAW = PROJECT_ROOT / "data" / "features"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

MERGED_FILE = DATA_RAW / "merged_features.csv"
CLEAN_MASTER_OUT = DATA_PROCESSED / "clean_master_data.csv"
NUMERIC_OUT = DATA_PROCESSED / "numeric_features_preprocessed.csv"
QUALITY_REPORT_OUT = DATA_PROCESSED / "data_quality_report.csv"
SUMMARY_OUT = DATA_PROCESSED / "preprocessing_summary.txt"

# --- Column groups (v2 schema: 88 columns) ---
METADATA_COLS = ["id", "file_name", "user_no", "question_id", "question"]

TARGET_COLS = [
    "openness", "conscientiousness", "extraversion", "agreeableness",
    "neuroticism", "overall_personality",
    "interview_score", "answer_score", "speaking_skills",
    "confidence_score", "facial_expression", "overall_performance"
]

# Audio prosodic (original)
AUDIO_COLS = [
    "Duration_Sec", "Speech_Rate_WPM", "Mean_Pitch_Hz",
    "Mean_Energy", "Mean_ZCR"
]

# Audio prosodic (new in v2)
AUDIO_COLS_NEW = [
    "Total_Words", "Articulation_Rate_WPM", "Total_Silence_Sec",
    "Silence_Ratio", "Pause_Count", "Avg_Pause_Duration_Sec",
    "Std_Pitch_Hz", "Pitch_Range_Hz", "Std_Energy",
    "Spectral_Centroid", "Spectral_Rolloff", "Spectral_Contrast",
    "Filler_Word_Count", "Filler_Rate_Per_Min"
]

MFCC_COLS = [f"MFCC_{i}" for i in range(1, 14)]

FACE_COLS = [
    "face_detected_ratio", "gaze_ratio_mean", "gaze_deviation_mean",
    "gaze_stability_std", "smile_score_mean", "smile_score_std",
    "frown_score_mean", "frown_score_std", "eye_openness_mean",
    "eye_openness_std", "jaw_open_mean", "jaw_open_std",
    "brow_raise_mean", "brow_raise_std", "mouth_frown_mean", "mouth_frown_std"
]

EMOTION_COLS = [
    "emotion_happy_mean", "emotion_sad_mean", "emotion_angry_mean",
    "emotion_surprise_mean", "emotion_fear_mean", "emotion_disgust_mean",
    "emotion_neutral_mean"
]

POSTURE_COLS = [
    "head_centering_score_mean", "absolute_shoulder_slope_mean",
    "shoulder_slope_var", "shoulder_width_mean", "shoulder_width_var",
    "nose_shoulder_dist_mean", "nose_shoulder_dist_var",
    "hand_speed_mean", "hand_to_face_touches", "crossed_arms_score",
    "core_speed_mean", "posture_shift_count", "engagement_score", "agitation_score"
]

TEXT_COLS = ["transcript"]

ALL_AUDIO = AUDIO_COLS + AUDIO_COLS_NEW
ALL_EXPECTED = (METADATA_COLS + TARGET_COLS + ALL_AUDIO + MFCC_COLS +
                FACE_COLS + EMOTION_COLS + POSTURE_COLS + TEXT_COLS)

print(f"Project root   : {PROJECT_ROOT}")
print(f"Input file     : {MERGED_FILE}")
print(f"Output dir     : {DATA_PROCESSED}")
print(f"Expected cols  : {len(ALL_EXPECTED)} (from schema)")
print(f"Dataset        : 2011 rows x 88 columns (v2)")

Project root   : C:\Users\ma983\Interview-Performance-Analyzer
Input file     : C:\Users\ma983\Interview-Performance-Analyzer\data\features\merged_features.csv
Output dir     : C:\Users\ma983\Interview-Performance-Analyzer\data\processed
Expected cols  : 87 (from schema)
Dataset        : 2011 rows x 88 columns (v2)


## 3. Load Dataset

In [3]:
# Load dataset
df = pd.read_csv(MERGED_FILE)

print(f"Dataset loaded successfully.")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

Dataset loaded successfully.
Shape: 2011 rows x 76 columns
Memory usage: 2.30 MB


## 4. Dataset Overview

In [4]:
# Basic overview
print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
print(f"Rows           : {df.shape[0]}")
print(f"Columns        : {df.shape[1]}")
print(f"Dtypes:")
for dtype, count in df.dtypes.value_counts().items():
    print(f"  {str(dtype):<20} {count}")
print()
print("First 5 column names:")
for i, col in enumerate(df.columns[:5], 1):
    print(f"  {i}. {col} ({df[col].dtype})")
print("...")
print(f"Last 5 column names:")
for i, col in enumerate(df.columns[-5:], len(df.columns)-4):
    print(f"  {i}. {col} ({df[col].dtype})")

DATASET OVERVIEW
Rows           : 2011
Columns        : 76
Dtypes:
  float64              65
  int64                6
  str                  5

First 5 column names:
  1. id (int64)
  2. file_name (str)
  3. duration_label (str)
  4. question_id (int64)
  5. question (str)
...
Last 5 column names:
  72. core_speed_mean (float64)
  73. posture_shift_count (int64)
  74. engagement_score (float64)
  75. agitation_score (int64)
  76. transcript (str)


## 5. Schema Audit

Generate a comprehensive audit of every column: dtype, unique count, missing values,
infinite values, constant detection, and numeric/categorical classification.

In [5]:
# Schema audit
def schema_audit(dataframe):
    audit_rows = []
    for col in dataframe.columns:
        series = dataframe[col]
        n_unique = series.nunique()
        missing = series.isnull().sum()
        missing_pct = missing / len(series) * 100

        if pd.api.types.is_numeric_dtype(series):
            inf_count = int(np.isinf(series).sum())
        else:
            inf_count = 0

        is_constant = (n_unique <= 1)

        is_nzv = False
        if pd.api.types.is_numeric_dtype(series):
            s_std = series.std()
            s_mean = series.mean()
            if s_std is not None and s_mean is not None and s_std > 0:
                cv = abs(s_std / s_mean) if s_mean != 0 else float('inf')
                is_nzv = cv < 0.01

        is_numeric = pd.api.types.is_numeric_dtype(series)
        is_cat = pd.api.types.is_object_dtype(series)

        audit_rows.append({
            'column': col,
            'dtype': str(series.dtype),
            'unique_count': n_unique,
            'missing_count': missing,
            'missing_percent': round(missing_pct, 2),
            'infinite_count': inf_count,
            'is_constant': is_constant,
            'is_near_zero_var': is_nzv,
            'is_numeric': is_numeric,
            'is_categorical': is_cat,
            'min': series.min() if is_numeric else None,
            'max': series.max() if is_numeric else None,
            'mean': series.mean() if is_numeric else None,
            'std': series.std() if is_numeric else None,
        })

    return pd.DataFrame(audit_rows)

audit = schema_audit(df)
print("Schema audit complete.")
print(f"Total columns: {len(audit)}")
print()

print("Dtype distribution:")
print(audit['dtype'].value_counts().to_string())
print()
print(f"Numeric columns     : {audit['is_numeric'].sum()}")
print(f"Categorical columns : {audit['is_categorical'].sum()}")
print(f"Constant columns    : {audit['is_constant'].sum()}")
print(f"Near-zero var cols  : {audit['is_near_zero_var'].sum()}")
print(f"Cols with missing   : {(audit['missing_count'] > 0).sum()}")
print(f"Cols with inf       : {(audit['infinite_count'] > 0).sum()}")

Schema audit complete.
Total columns: 76

Dtype distribution:
dtype
float64    65
int64       6
str         5

Numeric columns     : 71
Categorical columns : 0
Constant columns    : 0
Near-zero var cols  : 0
Cols with missing   : 1
Cols with inf       : 0


In [6]:
# Show columns with missing values
missing_audit = audit[audit['missing_count'] > 0].sort_values('missing_count', ascending=False)
if len(missing_audit) > 0:
    print("Columns with missing values:")
    print(missing_audit[['column', 'dtype', 'missing_count', 'missing_percent']].to_string(index=False))
else:
    print("No columns with missing values.")

Columns with missing values:
    column dtype  missing_count  missing_percent
transcript   str             13           0.6500


In [7]:
# Show columns with infinite values
inf_audit = audit[audit['infinite_count'] > 0]
if len(inf_audit) > 0:
    print("Columns with infinite values:")
    print(inf_audit[['column', 'dtype', 'infinite_count']].to_string(index=False))
else:
    print("No columns with infinite values.")

No columns with infinite values.


In [8]:
# Save the data quality report
audit.to_csv(QUALITY_REPORT_OUT, index=False)
print(f"Data quality report saved to: {QUALITY_REPORT_OUT}")

Data quality report saved to: C:\Users\ma983\Interview-Performance-Analyzer\data\processed\data_quality_report.csv


## 6. Column Grouping

Identify and classify all columns into known groups. Flag any expected-but-missing
or unknown columns.

In [9]:
# Column grouping
actual_cols = set(df.columns)

def check_group(name, expected):
    present = [c for c in expected if c in actual_cols]
    missing = [c for c in expected if c not in actual_cols]
    print(f"\n{'='*50}")
    print(f"  {name} ({len(present)}/{len(expected)} present)")
    print(f"{'='*50}")
    if present:
        print(f"  Columns: {present}")
    if missing:
        print(f"  WARNING - MISSING: {missing}")
    return present, missing

all_present = []
all_missing = []

for name, cols in [
    ("METADATA", METADATA_COLS),
    ("TARGETS", TARGET_COLS),
    ("AUDIO", AUDIO_COLS),
    ("MFCC", MFCC_COLS),
    ("FACE", FACE_COLS),
    ("EMOTION", EMOTION_COLS),
    ("POSTURE", POSTURE_COLS),
    ("TEXT", ["transcript"]),
]:
    present, missing = check_group(name, cols)
    all_present.extend(present)
    all_missing.extend(missing)

unknown = actual_cols - set(ALL_EXPECTED)

print(f"\n{'='*50}")
print(f"  UNKNOWN COLUMNS ({len(unknown)})")
print(f"{'='*50}")
if unknown:
    for c in sorted(unknown):
        print(f"  {c} (dtype={df[c].dtype})")
else:
    print("  None.")


  METADATA (5/5 present)
  Columns: ['id', 'file_name', 'user_no', 'question_id', 'question']

  TARGETS (12/12 present)
  Columns: ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance']

  AUDIO (5/5 present)
  Columns: ['Duration_Sec', 'Speech_Rate_WPM', 'Mean_Pitch_Hz', 'Mean_Energy', 'Mean_ZCR']

  MFCC (13/13 present)
  Columns: ['MFCC_1', 'MFCC_2', 'MFCC_3', 'MFCC_4', 'MFCC_5', 'MFCC_6', 'MFCC_7', 'MFCC_8', 'MFCC_9', 'MFCC_10', 'MFCC_11', 'MFCC_12', 'MFCC_13']

  FACE (16/16 present)
  Columns: ['face_detected_ratio', 'gaze_ratio_mean', 'gaze_deviation_mean', 'gaze_stability_std', 'smile_score_mean', 'smile_score_std', 'frown_score_mean', 'frown_score_std', 'eye_openness_mean', 'eye_openness_std', 'jaw_open_mean', 'jaw_open_std', 'brow_raise_mean', 'brow_raise_std', 'mouth_frown_mean', 'mouth_frown_std']

  EMOTIO

## 7. Missing Value Analysis

In [10]:
# Missing value analysis
print("=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_report = []
for col in df.columns:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        missing_report.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'missing_count': n_miss,
            'missing_percent': round(n_miss / len(df) * 100, 2),
            'group': 'TARGET' if col in TARGET_COLS else
                     'METADATA' if col in METADATA_COLS else
                     'TEXT' if col in TEXT_COLS else 'FEATURE'
        })

if missing_report:
    miss_df = pd.DataFrame(missing_report).sort_values('missing_count', ascending=False)
    print(miss_df.to_string(index=False))
else:
    print("No missing values found.")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

MISSING VALUE ANALYSIS
    column dtype  missing_count  missing_percent group
transcript   str             13           0.6500  TEXT

Total missing values: 13


In [11]:
# Strategy for missing values
print("\n" + "=" * 70)
print("MISSING VALUE STRATEGY")
print("=" * 70)

n_miss_transcript = df['transcript'].isnull().sum()
print(f"\ntranscript: {n_miss_transcript} missing ({n_miss_transcript/len(df)*100:.2f}%)")
print("  Strategy: KEEP NaN -- NLP pipeline will handle missing transcripts later.")
print("  Do NOT impute text with filler values.")

# Check numeric features for missing
feature_cols = [c for c in df.columns
                if c not in TARGET_COLS + METADATA_COLS + TEXT_COLS + ["video_quality"]
                and pd.api.types.is_numeric_dtype(df[c])]
miss_in_numeric = [(c, df[c].isnull().sum()) for c in feature_cols if df[c].isnull().sum() > 0]
if miss_in_numeric:
    print("\nMissing in numeric features:")
    for c, n in miss_in_numeric:
        print(f"  {c}: {n} missing")
else:
    print("\nNo missing values in numeric feature columns.")

# Check targets
miss_in_targets = [(c, df[c].isnull().sum()) for c in TARGET_COLS
                   if c in df.columns and df[c].isnull().sum() > 0]
if miss_in_targets:
    print("\nWARNING - Missing in TARGET columns (do NOT impute):")
    for c, n in miss_in_targets:
        print(f"  {c}: {n} missing ({n/len(df)*100:.2f}%)")
else:
    print("\nNo missing values in target columns.")


MISSING VALUE STRATEGY

transcript: 13 missing (0.65%)
  Strategy: KEEP NaN -- NLP pipeline will handle missing transcripts later.
  Do NOT impute text with filler values.

No missing values in numeric feature columns.

No missing values in target columns.


## 8. Duplicate Analysis

In [12]:
# Duplicate analysis
print("=" * 70)
print("DUPLICATE ANALYSIS")
print("=" * 70)

n_dup_rows = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dup_rows}")

print(f"\n--- id column ---")
print(f"  Total rows     : {len(df)}")
print(f"  Unique ids     : {df['id'].nunique()}")
print(f"  Duplicate ids  : {df.shape[0] - df['id'].nunique()}")

id_dup = df[df.duplicated(subset=['id'], keep=False)]
if len(id_dup) > 0:
    print(f"  WARNING - Found {len(id_dup)} rows with duplicate ids:")
    print(id_dup[['id', 'user_no', 'question_id']].head(10))
else:
    print(f"  OK - All ids are unique -- no duplicates.")

DUPLICATE ANALYSIS

Duplicate rows: 0

--- id column ---
  Total rows     : 2011
  Unique ids     : 2011
  Duplicate ids  : 0
  OK - All ids are unique -- no duplicates.


## 9. Identity / Candidate Analysis

`user_no` identifies the candidate. The same candidate appears in multiple rows
because they answer multiple questions. This is EXPECTED and should NOT be treated
as duplicates.

In [13]:
# Candidate analysis
print("=" * 70)
print("CANDIDATE (user_no) ANALYSIS")
print("=" * 70)

user_counts = df.groupby("user_no").size()

print(f"\nUnique candidates (user_no)  : {df['user_no'].nunique()}")
print(f"Total rows                   : {len(df)}")
print(f"\nVideos per candidate:")
print(user_counts.describe().to_string())

print(f"\nDistribution of videos per candidate:")
for n_vids in sorted(user_counts.unique()):
    n_users = (user_counts == n_vids).sum()
    print(f"  {n_vids:>3} videos : {n_users:>4} candidates")

print(f"\nTop 10 candidates by video count:")
print(user_counts.sort_values(ascending=False).head(10).to_string())

CANDIDATE (user_no) ANALYSIS

Unique candidates (user_no)  : 331
Total rows                   : 2011

Videos per candidate:
count   331.0000
mean      6.0755
std       1.5266
min       1.0000
25%       6.0000
50%       6.0000
75%       6.0000
max      17.0000

Distribution of videos per candidate:
    1 videos :   11 candidates
    2 videos :    4 candidates
    3 videos :    2 candidates
    4 videos :    6 candidates
    5 videos :    8 candidates
    6 videos :  223 candidates
    7 videos :   60 candidates
    8 videos :    8 candidates
    9 videos :    4 candidates
   11 videos :    1 candidates
   12 videos :    3 candidates
   17 videos :    1 candidates

Top 10 candidates by video count:
user_no
163    17
318    12
222    12
296    12
314    11
212     9
223     9
326     9
331     9
48      8


## 10. Infinite / Invalid Value Analysis

In [14]:
# Infinite value analysis
print("=" * 70)
print("INFINITE / INVALID VALUE ANALYSIS")
print("=" * 70)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
inf_report = []

for col in numeric_cols:
    series = df[col]
    inf_count = int(np.isinf(series).sum())
    nan_count = int(series.isnull().sum())

    if inf_count > 0 or nan_count > 0:
        clean = series.replace([np.inf, -np.inf], np.nan)
        inf_report.append({
            'column': col,
            'inf_count': inf_count,
            'nan_count': nan_count,
            'min': clean.min(),
            'max': clean.max(),
            'mean': clean.mean(),
            'std': clean.std(),
        })

if inf_report:
    inf_df = pd.DataFrame(inf_report)
    print(inf_df.to_string(index=False))
else:
    print("No infinite values found in any numeric column.")

print("\n--- Audio Feature Range Check ---")
audio_check = ["Duration_Sec", "Speech_Rate_WPM", "Silence_Duration_Sec",
               "Mean_Pitch_Hz", "Mean_Energy", "Mean_ZCR"]
for col in audio_check:
    if col in df.columns:
        vals = df[col].dropna()
        n_neg = (vals < 0).sum()
        n_zero = (vals == 0).sum()
        print(f"  {col:<25} min={vals.min():>10.4f}  max={vals.max():>10.4f}  neg={n_neg}  zero={n_zero}")

INFINITE / INVALID VALUE ANALYSIS
No infinite values found in any numeric column.

--- Audio Feature Range Check ---
  Duration_Sec              min=    0.6000  max=   92.3400  neg=0  zero=0
  Speech_Rate_WPM           min=    0.0000  max=  571.4300  neg=0  zero=13
  Silence_Duration_Sec      min=    0.0000  max=   29.4300  neg=0  zero=23
  Mean_Pitch_Hz             min=    0.0000  max= 1339.0700  neg=0  zero=7
  Mean_Energy               min=    0.0000  max=    0.2147  neg=0  zero=7
  Mean_ZCR                  min=    0.0000  max=    0.3371  neg=0  zero=1


## 11. Range Validation

Check domain-specific ranges. Probabilities/ratios should typically be [0, 1].
Audio features should be checked for impossible negatives. MFCC values are allowed
to be negative (spectral coefficients).

In [15]:
# Range validation
print("=" * 70)
print("RANGE VALIDATION")
print("=" * 70)

ratio_cols = [
    "face_detected_ratio", "gaze_ratio_mean", "smile_score_mean", "smile_score_std",
    "frown_score_mean", "frown_score_std", "eye_openness_mean", "eye_openness_std",
    "emotion_happy_mean", "emotion_sad_mean", "emotion_angry_mean",
    "emotion_surprise_mean", "emotion_fear_mean", "emotion_disgust_mean",
    "emotion_neutral_mean", "engagement_score", "agitation_score", "crossed_arms_score",
    "jaw_open_mean", "jaw_open_std", "brow_raise_mean", "brow_raise_std",
    "mouth_frown_mean", "mouth_frown_std"
]

print("\n--- Ratio/Probability Columns (expected [0, 1]) ---")
range_issues = []
for col in ratio_cols:
    if col in df.columns:
        vals = df[col].dropna()
        below_zero = (vals < 0).sum()
        above_one = (vals > 1).sum()
        status = "OK" if (below_zero == 0 and above_one == 0) else "FLAG"
        if status == "FLAG":
            range_issues.append({'column': col, 'below_0': below_zero, 'above_1': above_one})
        print(f"  {col:<35} min={vals.min():>8.4f}  max={vals.max():>8.4f}  [{status}]")

if range_issues:
    print(f"\nWARNING - {len(range_issues)} columns have values outside [0, 1]:")
    for r in range_issues:
        print(f"  {r['column']}: {r['below_0']} values < 0, {r['above_1']} values > 1")
else:
    print("\nAll ratio columns are within [0, 1].")

RANGE VALIDATION

--- Ratio/Probability Columns (expected [0, 1]) ---
  face_detected_ratio                 min=  0.0357  max=  1.0000  [OK]
  gaze_ratio_mean                     min=  0.4626  max=  0.5999  [OK]
  smile_score_mean                    min=  0.0000  max=  0.7666  [OK]
  smile_score_std                     min=  0.0000  max=  0.3959  [OK]
  frown_score_mean                    min=  0.0000  max=  0.5320  [OK]
  frown_score_std                     min=  0.0000  max=  0.2134  [OK]
  eye_openness_mean                   min=  0.3417  max=  0.9895  [OK]
  eye_openness_std                    min=  0.0000  max=  0.2926  [OK]
  emotion_happy_mean                  min=  0.0000  max=  0.6598  [OK]
  emotion_sad_mean                    min=  0.0005  max=  0.3615  [OK]
  emotion_angry_mean                  min=  0.0008  max=  0.2701  [OK]
  emotion_surprise_mean               min=  0.0053  max=  0.4036  [OK]
  emotion_fear_mean                   min=  0.0017  max=  0.2662  [OK]
  emoti

In [16]:
# Target range validation
print("\n--- Target Variable Ranges ---")
for col in TARGET_COLS:
    if col in df.columns:
        vals = df[col].dropna()
        print(f"  {col:<25} min={vals.min():>8.4f}  max={vals.max():>8.4f}  "
              f"mean={vals.mean():>8.4f}  std={vals.std():>8.4f}")

print("\n  Note: Target columns appear z-normalised (mean ~ 0, std ~ 1).")
print("  This is expected for the Big Five personality traits and interview scores.")


--- Target Variable Ranges ---
  openness                  min= -7.7996  max=  9.3388  mean= -0.0000  std=  1.1266
  conscientiousness         min= -6.9287  max=  6.6170  mean= -0.0000  std=  0.8770
  extraversion              min= -7.1844  max=  8.9102  mean= -0.0000  std=  1.0981
  agreeableness             min= -8.4123  max=  9.2420  mean= -0.0000  std=  1.2003
  neuroticism               min= -2.5766  max=  2.2259  mean=  0.0000  std=  0.4869
  overall_personality       min= -7.5262  max=  9.2695  mean= -0.0000  std=  1.2034
  interview_score           min= -7.8603  max=  9.3860  mean=  0.0000  std=  1.2301
  answer_score              min=-10.1991  max=  8.7223  mean=  0.0000  std=  1.1755
  speaking_skills           min= -9.4063  max=  7.9015  mean=  0.0000  std=  1.2776
  confidence_score          min= -7.2861  max=  6.8394  mean=  0.0000  std=  1.0797
  facial_expression         min= -7.4614  max=  9.2558  mean=  0.0000  std=  1.1510
  overall_performance       min= -9.3111  ma

In [17]:
# MFCC range validation
print("\n--- MFCC Feature Ranges (allowed to be negative) ---")
for col in MFCC_COLS:
    if col in df.columns:
        vals = df[col].dropna()
        print(f"  {col:<12} min={vals.min():>8.4f}  max={vals.max():>8.4f}  "
              f"mean={vals.mean():>8.4f}  std={vals.std():>8.4f}")

print("\n  MFCC values are spectral coefficients -- negative values are normal.")


--- MFCC Feature Ranges (allowed to be negative) ---
  MFCC_1       min=-1131.3710  max=-154.3277  mean=-408.9887  std= 60.3134
  MFCC_2       min=  0.0000  max=189.8129  mean=109.4728  std= 19.7687
  MFCC_3       min=-77.1459  max= 63.7223  mean= 19.0417  std= 18.0127
  MFCC_4       min=-28.3304  max= 41.8374  mean=  9.1694  std= 10.8499
  MFCC_5       min=-19.1488  max= 41.4020  mean= 14.7061  std=  9.5299
  MFCC_6       min=-23.5699  max= 47.9431  mean=  9.7946  std=  9.6804
  MFCC_7       min=-31.0600  max= 17.2124  mean= -7.5293  std=  7.3981
  MFCC_8       min=-26.4962  max= 26.3996  mean=  1.0121  std= 10.0004
  MFCC_9       min=-32.2217  max= 14.9128  mean= -8.9946  std=  6.0102
  MFCC_10      min=-26.9651  max= 17.7082  mean= -4.4348  std=  8.1863
  MFCC_11      min=-21.6639  max= 12.9527  mean= -5.2624  std=  5.1544
  MFCC_12      min=-26.3776  max= 11.4072  mean= -2.8786  std=  5.5555
  MFCC_13      min=-24.6556  max= 10.1096  mean= -5.8545  std=  5.0979

  MFCC values are 

## 12. Categorical Variable Analysis

In [18]:
# Categorical analysis
print("=" * 70)
print("CATEGORICAL VARIABLE ANALYSIS")
print("=" * 70)

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject dtype columns: {cat_cols}")

for col in cat_cols:
    n_unique = df[col].nunique()
    print(f"\n--- {col} ---")
    print(f"  Unique values: {n_unique}")
    print(f"  Missing: {df[col].isnull().sum()}")
    if n_unique <= 20:
        print(f"  Value counts:")
        print(df[col].value_counts().to_string())
    else:
        print(f"  Top 10 values:")
        print(df[col].value_counts().head(10).to_string())

CATEGORICAL VARIABLE ANALYSIS

Object dtype columns: ['file_name', 'duration_label', 'question', 'video_quality', 'transcript']

--- file_name ---
  Unique values: 2011
  Missing: 0
  Top 10 values:
file_name
vid_0001.mp4    1
vid_0002.mp4    1
vid_0003.mp4    1
vid_0004.mp4    1
vid_0005.mp4    1
vid_0006.mp4    1
vid_0007.mp4    1
vid_0008.mp4    1
vid_0009.mp4    1
vid_0010.mp4    1

--- duration_label ---
  Unique values: 3
  Missing: 0
  Value counts:
duration_label
medium    754
short     680
long      577

--- question ---
  Unique values: 76
  Missing: 0
  Top 10 values:
question
Introduce yourself                                                                                193
Tell me about yourself                                                                            192
Are you open to take risks? or Do you like experimenting?                                          46
How would you rate yourself on a scale of 1 to 10?                                                 

In [19]:
# Categorical encoding plan
print("\n" + "=" * 70)
print("CATEGORICAL ENCODING PLAN")
print("=" * 70)

encoding_plan = (
    "\nduration_label:\n"
    "  - Check if ordinal (e.g., short < medium < long)\n"
    "  - If ordinal: map to ordered integers\n"
    "  - If nominal: one-hot encode\n"
    "\nquestion_id:\n"
    "  - Preserve as categorical metadata\n"
    "  - Do NOT label-encode as ML feature\n"
    "  - Useful for question-level analysis\n"
    "\nuser_no:\n"
    "  - Do NOT label-encode as ML feature\n"
    "  - Use for candidate-level aggregation\n"
    "  - Use for GroupKFold splitting\n"
    "  - Preserve as grouping variable\n"
    "\nfile_name:\n"
    "  - Metadata only -- not a feature\n"
    "  - Useful for traceability"
)
print(encoding_plan)


CATEGORICAL ENCODING PLAN

duration_label:
  - Check if ordinal (e.g., short < medium < long)
  - If ordinal: map to ordered integers
  - If nominal: one-hot encode

question_id:
  - Preserve as categorical metadata
  - Do NOT label-encode as ML feature
  - Useful for question-level analysis

user_no:
  - Do NOT label-encode as ML feature
  - Use for candidate-level aggregation
  - Use for GroupKFold splitting
  - Preserve as grouping variable

file_name:
  - Metadata only -- not a feature
  - Useful for traceability


## 13. Target Variable Validation

In [20]:
# Target validation
print("=" * 70)
print("TARGET VARIABLE VALIDATION")
print("=" * 70)

for col in TARGET_COLS:
    if col in df.columns:
        vals = df[col]
        print(f"\n--- {col} ---")
        print(f"  dtype    : {vals.dtype}")
        print(f"  non-null : {vals.notna().sum()} / {len(vals)}")
        print(f"  missing  : {vals.isnull().sum()}")
        print(f"  unique   : {vals.nunique()}")
        if pd.api.types.is_numeric_dtype(vals):
            print(f"  min      : {vals.min():.4f}")
            print(f"  max      : {vals.max():.4f}")
            print(f"  mean     : {vals.mean():.4f}")
            print(f"  std      : {vals.std():.4f}")
            print(f"  skewness : {vals.skew():.4f}")

# Check target-target correlations
print("\n--- Target Correlation Matrix ---")
target_corr = df[TARGET_COLS].corr()
print(target_corr.round(3).to_string())

TARGET VARIABLE VALIDATION

--- openness ---
  dtype    : float64
  non-null : 2011 / 2011
  missing  : 0
  unique   : 2011
  min      : -7.7996
  max      : 9.3388
  mean     : -0.0000
  std      : 1.1266
  skewness : 0.0266

--- conscientiousness ---
  dtype    : float64
  non-null : 2011 / 2011
  missing  : 0
  unique   : 2011
  min      : -6.9287
  max      : 6.6170
  mean     : -0.0000
  std      : 0.8770
  skewness : -0.5699

--- extraversion ---
  dtype    : float64
  non-null : 2011 / 2011
  missing  : 0
  unique   : 2011
  min      : -7.1844
  max      : 8.9102
  mean     : -0.0000
  std      : 1.0981
  skewness : -0.2183

--- agreeableness ---
  dtype    : float64
  non-null : 2011 / 2011
  missing  : 0
  unique   : 2011
  min      : -8.4123
  max      : 9.2420
  mean     : -0.0000
  std      : 1.2003
  skewness : 0.3976

--- neuroticism ---
  dtype    : float64
  non-null : 2011 / 2011
  missing  : 0
  unique   : 2011
  min      : -2.5766
  max      : 2.2259
  mean     : 0.0

## 14. Data Leakage Audit

Check whether any feature is directly derived from a target, duplicate of a target,
or suspiciously identical to a target.

In [21]:
# Leakage audit
print("=" * 70)
print("DATA LEAKAGE AUDIT")
print("=" * 70)

feature_cols = [c for c in df.columns
                if c not in TARGET_COLS + METADATA_COLS + TEXT_COLS + ["video_quality"]]

# Check feature-target correlations
print("\nFeature-Target correlations (|r| > 0.5 flagged):")
print("-" * 70)
suspicious = []
for feat in feature_cols:
    if pd.api.types.is_numeric_dtype(df[feat]):
        for tgt in TARGET_COLS:
            if tgt in df.columns:
                corr = df[feat].corr(df[tgt])
                if abs(corr) > 0.5:
                    suspicious.append({
                        'feature': feat,
                        'target': tgt,
                        'correlation': round(corr, 4)
                    })

if suspicious:
    susp_df = pd.DataFrame(suspicious).sort_values('correlation', key=abs, ascending=False)
    print(susp_df.to_string(index=False))
    print("\nNote: High correlation does not automatically mean leakage.")
    print("Investigate whether the feature is derived from the target.")
else:
    print("No suspiciously high feature-target correlations found.")

# Check for duplicate columns between features and targets
print("\n--- Exact duplicate check between features and targets ---")
for feat in feature_cols:
    for tgt in TARGET_COLS:
        if tgt in df.columns and feat in df.columns:
            if df[feat].equals(df[tgt]):
                print(f"  EXACT MATCH: {feat} == {tgt}")

print("\nLeakage audit complete. No automatic deletions performed.")

DATA LEAKAGE AUDIT

Feature-Target correlations (|r| > 0.5 flagged):
----------------------------------------------------------------------


No suspiciously high feature-target correlations found.

--- Exact duplicate check between features and targets ---

Leakage audit complete. No automatic deletions performed.


## 15. Outlier Detection Report

Do NOT automatically delete outliers. Interview behavioral data may have genuine
extreme values representing high/low performers.

In [22]:
# Outlier detection using IQR method
print("=" * 70)
print("OUTLIER DETECTION REPORT (IQR method)")
print("=" * 70)

outlier_report = []
for col in feature_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        series = df[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_outliers = ((series < lower) | (series > upper)).sum()
        pct_outliers = n_outliers / len(series) * 100
        if n_outliers > 0:
            outlier_report.append({
                'column': col,
                'n_outliers': n_outliers,
                'pct_outliers': round(pct_outliers, 2),
                'lower_fence': round(lower, 4),
                'upper_fence': round(upper, 4),
                'min': round(series.min(), 4),
                'max': round(series.max(), 4),
            })

outlier_df = pd.DataFrame(outlier_report).sort_values('pct_outliers', ascending=False)
print(f"\nFeatures with outliers (IQR method):")
print(outlier_df.to_string(index=False))

print("\nNote: Outliers are flagged but NOT removed.")
print("Extreme values may represent genuine behavioral variation.")

OUTLIER DETECTION REPORT (IQR method)



Features with outliers (IQR method):
                      column  n_outliers  pct_outliers  lower_fence  upper_fence        min       max
             agitation_score         274       13.6300     -12.0000      20.0000     0.0000  299.0000
         posture_shift_count         274       13.6300     -12.0000      20.0000     0.0000  299.0000
            mouth_frown_mean         270       13.4300      -0.0032       0.0056     0.0000    0.0955
            frown_score_mean         246       12.2300      -0.0435       0.0772     0.0000    0.5320
             mouth_frown_std         242       12.0300      -0.0056       0.0099     0.0000    0.1378
          shoulder_width_var         225       11.1900      -0.0002       0.0005     0.0000    0.0126
          emotion_angry_mean         225       11.1900      -0.0202       0.0492     0.0008    0.2701
          emotion_happy_mean         211       10.4900      -0.0962       0.1710     0.0000    0.6598
            smile_score_mean         200    

## 16. Preprocessing Transformations

Based on the audit above, apply minimal preprocessing:
- No rows are dropped (all data is valid)
- No columns are dropped (feature selection happens later)
- No scaling is applied (scaling happens after train/test split)
- No imputation is applied to transcript (NLP pipeline handles it)
- Categorical columns are preserved as-is for now

In [23]:
# Create a copy for any transformations
df_clean = df.copy()

print("Preprocessing decisions:")
print("  - No rows removed (all 2011 rows are valid)")
print("  - No columns removed (feature selection in later stage)")
print("  - No scaling applied (deferred to after train/test split)")
print("  - No NLP preprocessing (deferred to NLP pipeline)")
print("  - No imputation on transcript (deferred to NLP pipeline)")
print("  - No imputation on targets (13 missing transcripts only)")
print()
print(f"Clean shape: {df_clean.shape}")

Preprocessing decisions:
  - No rows removed (all 2011 rows are valid)
  - No columns removed (feature selection in later stage)
  - No scaling applied (deferred to after train/test split)
  - No NLP preprocessing (deferred to NLP pipeline)
  - No imputation on transcript (deferred to NLP pipeline)
  - No imputation on targets (13 missing transcripts only)

Clean shape: (2011, 76)


In [24]:
# Verify data types are correct
print("Data type verification:")
for col in TARGET_COLS:
    if col in df_clean.columns:
        print(f"  {col:<25} dtype={df_clean[col].dtype}  (should be numeric)")

print()
for col in ["id", "file_name", "user_no", "question_id", "question", "duration_label", "video_quality"]:
    if col in df_clean.columns:
        print(f"  {col:<25} dtype={df_clean[col].dtype}  (should be object/string)")

Data type verification:
  openness                  dtype=float64  (should be numeric)
  conscientiousness         dtype=float64  (should be numeric)
  extraversion              dtype=float64  (should be numeric)
  agreeableness             dtype=float64  (should be numeric)
  neuroticism               dtype=float64  (should be numeric)
  overall_personality       dtype=float64  (should be numeric)
  interview_score           dtype=float64  (should be numeric)
  answer_score              dtype=float64  (should be numeric)
  speaking_skills           dtype=float64  (should be numeric)
  confidence_score          dtype=float64  (should be numeric)
  facial_expression         dtype=float64  (should be numeric)
  overall_performance       dtype=float64  (should be numeric)

  id                        dtype=int64  (should be object/string)
  file_name                 dtype=str  (should be object/string)
  user_no                   dtype=int64  (should be object/string)
  question_id       

## 17. Save Processed Datasets

In [25]:
# Save clean master dataset
df_clean.to_csv(CLEAN_MASTER_OUT, index=False)
print(f"Clean master dataset saved to: {CLEAN_MASTER_OUT}")
print(f"  Shape: {df_clean.shape}")
print(f"  Columns: {list(df_clean.columns)}")

Clean master dataset saved to: C:\Users\ma983\Interview-Performance-Analyzer\data\processed\clean_master_data.csv
  Shape: (2011, 76)
  Columns: ['id', 'file_name', 'duration_label', 'question_id', 'question', 'video_quality', 'user_no', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance', 'Duration_Sec', 'Speech_Rate_WPM', 'Silence_Duration_Sec', 'Mean_Pitch_Hz', 'Mean_Energy', 'Mean_ZCR', 'MFCC_1', 'MFCC_2', 'MFCC_3', 'MFCC_4', 'MFCC_5', 'MFCC_6', 'MFCC_7', 'MFCC_8', 'MFCC_9', 'MFCC_10', 'MFCC_11', 'MFCC_12', 'MFCC_13', 'face_detected_ratio', 'gaze_ratio_mean', 'gaze_deviation_mean', 'gaze_stability_std', 'smile_score_mean', 'smile_score_std', 'frown_score_mean', 'frown_score_std', 'eye_openness_mean', 'eye_openness_std', 'jaw_open_mean', 'jaw_open_std', 'brow_raise_mean', 'brow_raise_std', 'mouth_frown_mean', 'mouth_

In [26]:
# Save numeric model-ready dataframe
# Exclude: raw transcript, question, file_name, id
# Keep: user_no (for grouping), all numeric features, targets
exclude_from_numeric = ["transcript", "question", "file_name", "id"]
numeric_df = df_clean.drop(columns=[c for c in exclude_from_numeric if c in df_clean.columns])

numeric_df.to_csv(NUMERIC_OUT, index=False)
print(f"Numeric features saved to: {NUMERIC_OUT}")
print(f"  Shape: {numeric_df.shape}")
print(f"  Excluded: {exclude_from_numeric}")

Numeric features saved to: C:\Users\ma983\Interview-Performance-Analyzer\data\processed\numeric_features_preprocessed.csv
  Shape: (2011, 72)
  Excluded: ['transcript', 'question', 'file_name', 'id']


## 18. Final Preprocessing Summary

In [27]:
# Final summary
print("=" * 70)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 70)

summary = (
    f"\nOriginal shape: {df.shape}"
    f"\nCleaned shape: {df_clean.shape}"
    f"\n\nRemoved rows: 0 (no rows were removed)"
    f"\nRemoved columns: 0 (no columns were removed)"
    f"\n\nNumeric columns: {df_clean.select_dtypes(include=[np.number]).shape[1]}"
    f"\nCategorical columns: {df_clean.select_dtypes(include=['object']).shape[1]}"
    f"\nText columns: 2 (transcript, question)"
    f"\n\nMissing values before: {df.isnull().sum().sum()}"
    f"\nMissing values after: {df_clean.isnull().sum().sum()}"
    f"\n\nDuplicate rows: {df.duplicated().sum()}"
    f"\nDuplicate IDs: {df.shape[0] - df['id'].nunique()}"
    f"\nUnique candidates: {df['user_no'].nunique()}"
    f"\nAverage videos per candidate: {round(df.groupby('user_no').size().mean(), 1)}"
    f"\n\nTargets: {TARGET_COLS}"
)
print(summary)

print("Files created:")
print(f"  1. {CLEAN_MASTER_OUT}")
print(f"  2. {NUMERIC_OUT}")
print(f"  3. {QUALITY_REPORT_OUT}")
print(f"  4. {SUMMARY_OUT}")

FINAL PREPROCESSING SUMMARY

Original shape: (2011, 76)
Cleaned shape: (2011, 76)

Removed rows: 0 (no rows were removed)
Removed columns: 0 (no columns were removed)

Numeric columns: 71
Categorical columns: 5
Text columns: 2 (transcript, question)

Missing values before: 13
Missing values after: 13

Duplicate rows: 0
Duplicate IDs: 0
Unique candidates: 331
Average videos per candidate: 6.1

Targets: ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance']
Files created:
  1. C:\Users\ma983\Interview-Performance-Analyzer\data\processed\clean_master_data.csv
  2. C:\Users\ma983\Interview-Performance-Analyzer\data\processed\numeric_features_preprocessed.csv
  3. C:\Users\ma983\Interview-Performance-Analyzer\data\processed\data_quality_report.csv
  4. C:\Users\ma983\Interview-Performance-Analyzer\data\processed\preprocessing

In [28]:
# Save preprocessing summary to text file
summary_lines = [
    "INTERVIEW PERFORMANCE ANALYZER - PREPROCESSING SUMMARY",
    "=" * 60,
    "",
    f"Original shape: {df.shape}",
    f"Cleaned shape: {df_clean.shape}",
    "",
    "Removed rows: 0 (no rows were removed)",
    "Removed columns: 0 (no columns were removed)",
    "",
    f"Numeric columns: {df_clean.select_dtypes(include=[np.number]).shape[1]}",
    f"Categorical columns: {df_clean.select_dtypes(include=['object']).shape[1]}",
    "Text columns: 2 (transcript, question)",
    "",
    f"Missing values before: {df.isnull().sum().sum()}",
    f"Missing values after: {df_clean.isnull().sum().sum()}",
    "",
    f"Duplicate rows: {df.duplicated().sum()}",
    f"Duplicate IDs: {df.shape[0] - df['id'].nunique()}",
    f"Unique candidates: {df['user_no'].nunique()}",
    f"Average videos per candidate: {round(df.groupby('user_no').size().mean(), 1)}",
    "",
    "Targets:",
] + [f"  - {t}" for t in TARGET_COLS] + [
    "",
    "WHAT WAS CHANGED:",
    "  - Schema audit generated and saved",
    "  - Column groups validated against reference schema",
    "  - Missing values documented (transcript: 13 rows)",
    "  - Duplicate check completed (0 duplicate rows, all IDs unique)",
    "  - Infinite/invalid value check completed",
    "  - Range validation completed for ratio columns",
    "  - Outlier detection completed (IQR method, flagged only)",
    "  - Data leakage audit completed",
    "",
    "WHAT WAS INTENTIONALLY NOT CHANGED:",
    "  - No rows dropped",
    "  - No columns dropped",
    "  - No scaling applied",
    "  - No feature engineering",
    "  - No PCA",
    "  - No NLP preprocessing",
    "  - No imputation on transcript",
    "  - No categorical encoding",
    "",
    "NEXT STEPS (EDA stage):",
    "  - Distribution analysis for all features",
    "  - Correlation heatmap analysis",
    "  - Feature-target relationship visualization",
    "  - Outlier visualization and decision",
    "  - Categorical encoding decision",
    "  - Video quality impact analysis",
    "",
    "LATER (Feature Engineering):",
    "  - Candidate-level aggregation (mean, std, min, max per user)",
    "  - Interaction features",
    "  - Composite target engineering",
    "  - NLP pipeline for transcripts",
    "  - Scaling after train/test split",
]

with open(SUMMARY_OUT, 'w') as f:
    f.write('\n'.join(summary_lines))

print(f"Preprocessing summary saved to: {SUMMARY_OUT}")

Preprocessing summary saved to: C:\Users\ma983\Interview-Performance-Analyzer\data\processed\preprocessing_summary.txt


In [29]:
# Final verification
print("=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

# Verify all expected files exist
for filepath in [CLEAN_MASTER_OUT, NUMERIC_OUT, QUALITY_REPORT_OUT, SUMMARY_OUT]:
    exists = filepath.exists()
    size = filepath.stat().st_size if exists else 0
    status = f"OK ({size:,} bytes)" if exists else "MISSING"
    print(f"  {filepath.name:<40} {status}")

print("\nPreprocessing pipeline complete.")

FINAL VERIFICATION
  clean_master_data.csv                    OK (3,150,805 bytes)
  numeric_features_preprocessed.csv        OK (2,069,042 bytes)
  data_quality_report.csv                  OK (9,434 bytes)
  preprocessing_summary.txt                OK (1,903 bytes)

Preprocessing pipeline complete.
